In [ ]:
import pandas as pd
from silver_staging_utils import connect_to_postgres, read_table
import numpy as np
import json
import re
import unicodedata

# Productos

In [ ]:
query = '''
SELECT
    *
FROM silver.products
WHERE
    snapshot_date IN (
        SELECT DISTINCT snapshot_date FROM silver.products ORDER BY snapshot_date DESC LIMIT 10   
    )
'''
print(query)

conn = connect_to_postgres()
if conn:
    df = read_table(query, conn)
    conn.close()

In [ ]:
df.info()

In [ ]:
df['snapshot_date'].unique()

In [ ]:
df['supermarket'].unique()

## Final Price creation

##### Price problem with Casa Rica

In [ ]:
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace(r"[^\d,\.]", "", regex=True)
    .str.replace(".", "", regex=False) 
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

##### Coalesce to build final price

In [ ]:
df.loc[df['promotion_price'] == '0', 'promotion_price'] = None

In [ ]:
df["final_price"] = (
    df["promotion_price"]
        .combine_first(df['price'])
)

# Categoria

In [ ]:
query = '''
SELECT
    *
FROM silver.categories
WHERE
    snapshot_date IN (
        SELECT MAX(snapshot_date) FROM silver.categories  
    )
'''
print(query)

conn = connect_to_postgres()
if conn:
    df = read_table(query, conn)
    conn.close()

In [ ]:
df.info()

In [ ]:
df['snapshot_date'].unique()

In [ ]:
df['supermarket'].unique()

### category_final_slug
Creacion de un slug homogeneo para todos los supermercados, a utilizar para la surrogate key

In [ ]:
mask_real = df['supermarket'] == 'real'

df.loc[mask_real, 'real_lvl1_clean'] = (
    df.loc[mask_real, 'category_lvl1_slug']
        .str.split('/', n=1)
        .str[-1]
)


In [ ]:
df['category_slug_final'] = (
    np.where(df['supermarket'] == 'biggie', df['category_lvl1_slug'],
    np.where(df['supermarket'] == 'casa rica', df['category_lvl2_slug'],
    np.where(df['supermarket'].isin(['stock', 'super seis']), df['category_lvl3_slug'],
    np.where(df['supermarket'] == 'real', df['real_lvl1_clean'],
             None))))
)

In [ ]:
df.head(1000)

### category_final_name

Análisis

In [ ]:
query = '''
SELECT
    supermarket,
	category_lvl1_name
FROM silver.categories
'''
print(query)

conn = connect_to_postgres()
if conn:
    df = read_table(query, conn)
    conn.close()

In [ ]:
df.head()

In [ ]:
pivot = (
    df
    .assign(value=True)          # Marcamos presencia
    .pivot_table(
        index='category_lvl1_name',
        columns='supermarket',
        values='value',
        aggfunc='any',           # Si existe al menos 1 → True
        fill_value=False
    )
)

pivot_int = pivot.astype(int)


In [ ]:
# pivot_int.to_csv("/workspaces/tesis-ivan-gennaro/scripts/silver_staging/category_final_name_analisis.csv")

Aplicacion

In [ ]:
def normalize_text(s):
    """
    Normaliza una cadena para matching:
      - pasa a minúsculas
      - quita acentos
      - reemplaza caracteres no alfanuméricos por espacio
      - reduce espacios múltiples a uno
      - strip()
    """
    if s is None:
        return None
    s = str(s).strip().lower()
    # quitar acentos
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    # deja solo letras, números y espacios
    s = re.sub(r"[^a-z0-9\s]+", " ", s)
    # colapsar espacios
    s = re.sub(r"\s+", " ", s).strip()
    return s


In [ ]:
df['category_clean'] = df['category_lvl1_name'].apply(normalize_text)


In [ ]:

MAPPINGS_FILE = '/workspaces/tesis-ivan-gennaro/scripts/silver_staging/mapeo_categoria_final.json'

In [ ]:
# Leer mappings desde JSON
with open(MAPPINGS_FILE, 'r', encoding='utf-8') as f:
    mappings = json.load(f)

inv_maps = {sup: {} for sup in ["biggie", "casa_rica", "real", "s6", "stock"]}

for final, mapping in mappings.items():
    for sup, cats in mapping.items():
        if cats is None:
            continue
        
        # Normalizar: convertir string -> lista de strings
        if isinstance(cats, str):
            cats = [cats]

        # Normalizar cada valor
        cats_norm = [normalize_text(c) for c in cats]

        # Guardar mapping categoria_super → categoria_final
        for c in cats_norm:
            inv_maps[sup][c] = final   # final NO normalizado, mejor para presentación


In [ ]:
# 3) Mapear la categoría final
df['category_final'] = df.apply(
    lambda row: inv_maps[row['supermarket']].get(row['category_clean']),
    axis=1
)

In [ ]:
df.drop_duplicates().to_csv("/workspaces/tesis-ivan-gennaro/scripts/silver_staging/category_final_name.csv")

In [ ]:
df[df["category_final"].isna()][["supermarket", "category_lvl1_name", "category_clean"]].drop_duplicates()

In [ ]:
df["category_final"].drop_duplicates()